# Chapter 1 — Make the first model call for Lucy

A 90-minute practical class. **Write a prediction before running each experiment.** A correct prediction includes a reason; an incorrect prediction is useful evidence of what to investigate.

You are building for Lucy, who owns an ice cream shop. By the end you will construct a model request, reject malformed responses, expose the limits of a content check, and detect a changed shop snapshot.

This notebook is standalone: Python 3.11 or newer, standard library only, no install and no credentials needed. Check the displayed interpreter version; do not assume a hosted notebook's current version. The full book uses Python 3.14. Upload this `.ipynb` to Colab or open it in an existing Jupyter environment. Choose **Restart runtime/kernel and Run all** for a clean replay. The live experiment is OFF by default.

A model has learned knowledge from training. This request supplies its authoritative *current shop facts*; it cannot observe a later sale unless we provide that observation. A response is generated data, not proof that an order exists. No cell has a purchasing capability.

Keep three claims separate: **valid envelope**, **no problems found by this limited check**, and **facts verified against our records**. The second does not imply the third.

## Revision 2 — endpoint and response boundaries

This revision replaces the earlier classroom download for new use. It refuses HTTP redirects, so a configured model endpoint cannot redirect the helper to another destination with your credential. Both direct successes and failures remain explicit. The parser now requires the assistant message role; this checks the envelope shape, not authenticity or factual truth. Keep live calls disabled in the historical v1 notebook.


In [ ]:
import copy
import hashlib
import json
import os
import sys
from urllib.error import HTTPError, URLError
from urllib.parse import urlsplit
from urllib.request import HTTPRedirectHandler, Request, build_opener

assert sys.version_info >= (3, 11), "Use Python 3.11 or newer for this notebook"
print("Python", sys.version.split()[0], "— offline class, standard library only")

## 1. Establish the shop facts — 10 minutes

**PREDICT:** which SKUs need replenishment, and how many tubs does each need? At exactly the reorder point, should needed be zero or one? Write the three answers before running.

The reorder point is also the replenishment target in this teaching fixture. `needed` is a deterministic calculation, not a model decision. Money and ordering come later.

In [ ]:
SHOP = {
    "customer": "Lucy",
    "currency": "GBP",
    "products": [
        {"sku": "SKU-VANILLA", "name": "Vanilla", "on_hand": 2, "reorder_point": 8},
        {"sku": "SKU-CHOCOLATE", "name": "Chocolate", "on_hand": 12, "reorder_point": 6},
        {"sku": "SKU-STRAWBERRY", "name": "Strawberry", "on_hand": 1, "reorder_point": 5},
    ],
}


def stock_facts(shop):
    return [
        (p["sku"], p["on_hand"], max(0, p["reorder_point"] - p["on_hand"]))
        for p in sorted(shop["products"], key=lambda p: p["sku"])
    ]


print("SKU, physical stock, needed:")
for row in stock_facts(SHOP):
    print(row)
assert stock_facts(SHOP) == [
    ("SKU-CHOCOLATE", 12, 0),
    ("SKU-STRAWBERRY", 1, 4),
    ("SKU-VANILLA", 2, 6),
]

## 2. Inspect the whole request — 10 minutes

**PREDICT:** locate currency, stock counts, and the reorder rule in the request. Where is evidence of a supplier order? Which facts would change if Lucy sold one vanilla tub?

Run this cell and inspect the complete printed request. The instruction is guidance; the absence of a purchase function prevents actual ordering. Token limits and temperature do not establish truth.

In [ ]:
def messages(shop):
    return [
        {
            "role": "system",
            "content": "Write a short morning stock brief for Lucy. "
            "Use only the supplied stock facts. "
            "Name products below their reorder points. Do not purchase anything "
            "or claim that an order exists.",
        },
        {"role": "user", "content": json.dumps(shop, sort_keys=True)},
    ]


def payload(shop, model="your-model-id", max_tokens=256):
    return {
        "model": model,
        "messages": messages(shop),
        "stream": False,
        "temperature": 0,
        "max_tokens": max_tokens,
    }


request_body = payload(SHOP)
print(json.dumps(request_body, indent=2))
print("UTF-8 bytes:", len(json.dumps(request_body).encode("utf-8")))
assert json.loads(request_body["messages"][1]["content"]) == SHOP

## 3. Read a response envelope — 15 minutes

**PREDICT:** distinguish malformed JSON shape, a length-limited response, a tool request, and completed nonempty text. Which belongs in today's plain-text reader?

A function definition makes no network request. This reader returns text only when the envelope meets its contract. It has no evidence that the text is factually correct.

In [ ]:
def read_brief(document):
    if not isinstance(document, dict):
        raise ValueError("completion envelope must be an object")
    choices = document.get("choices")
    if not isinstance(choices, list) or len(choices) != 1:
        raise ValueError("one completion required")
    choice = choices[0]
    if not isinstance(choice, dict) or choice.get("finish_reason") != "stop":
        raise ValueError("completion did not finish normally")
    message = choice.get("message", {})
    if not isinstance(message, dict) or message.get("tool_calls") or message.get("refusal"):
        raise ValueError("a plain completed brief was expected")
    if message.get("role") != "assistant":
        raise ValueError("assistant completion role required")
    text = message.get("content")
    if not isinstance(text, str) or not text.strip():
        raise ValueError("nonempty brief required")
    return text


OFFLINE_RESPONSE = {
    "choices": [
        {
            "finish_reason": "stop",
            "message": {
                "role": "assistant",
                "content": "Vanilla has 2 tubs and strawberry has 1: "
                "both are below their reorder points. "
                "Chocolate has 12 tubs, above its reorder point. No orders have been placed.",
            },
        }
    ]
}
print(read_brief(OFFLINE_RESPONSE))

In [ ]:
# PREDICT a rejection reason for every entry before running.
BROKEN = {
    "not an object": [],
    "missing choice": {"error": {"message": "rate limited"}},
    "non-object choice": {"choices": [17]},
    "two choices": {"choices": [{}, {}]},
    "truncated": {"choices": [{"finish_reason": "length", "message": {"content": "Vanilla"}}]},
    "tool request": {"choices": [{"finish_reason": "stop", "message": {"tool_calls": [{}]}}]},
    "empty text": {"choices": [{"finish_reason": "stop", "message": {"content": " "}}]},
}
for name, document in BROKEN.items():
    try:
        read_brief(document)
    except ValueError as error:
        print(name, "-> REJECTED:", error)
    else:
        raise AssertionError("Unexpected acceptance: " + name)

## 4. Challenge a plausible answer, then challenge the checker — 20 minutes

**PREDICT:** will the envelope reader reject the claimed purchase below? Will the content check flag it? Then invent a false sentence that the content check misses.

`check_brief` is a deliberately small **warning heuristic**. It finds missing product names and a few phrases; it cannot verify arbitrary English, quantities, negation or supplier activity. An empty list means **no flags found**, never “true” or “safe to act”. The experiment must expose both a useful detection and a false negative.

In [ ]:
def check_brief(text, shop):
    lower = text.lower()
    problems = [
        "omits low product " + p["name"]
        for p in shop["products"]
        if p["on_hand"] < p["reorder_point"] and p["name"].lower() not in lower
    ]
    for phrase in ("placed the supplier order", "placed an order", "purchased"):
        if phrase in lower:
            problems.append("possible unsupported action: " + phrase)
    return problems


LYING_RESPONSE = copy.deepcopy(OFFLINE_RESPONSE)
LYING_RESPONSE["choices"][0]["message"]["content"] = (
    "Vanilla and strawberry need attention. I placed the supplier order."
)
for label, document in [
    ("authored fixture", OFFLINE_RESPONSE),
    ("claimed purchase", LYING_RESPONSE),
]:
    text = read_brief(document)
    print(label, "->", check_brief(text, SHOP) or "NO FLAGS — still needs factual review")

MISSED_LIE = "Vanilla has 200 tubs; strawberry has 100. The supplier confirmed our purchase."
assert check_brief(MISSED_LIE, SHOP) == []
print("Missed false claim:", MISSED_LIE)
print("Independent current stock:", stock_facts(SHOP))
NEGATED_ACTION = "Vanilla and strawberry are low. I have not purchased anything."
print("False positive:", check_brief(NEGATED_ACTION, SHOP))

**Discuss:** why does adding one more banned phrase fail to make this a general truth checker? If the teacher says “the check passed, therefore believe the answer”, use `MISSED_LIE` as a counterexample.

A dependable stock display can use `stock_facts(SHOP)` directly and label model prose as a draft. This is a useful scripted baseline. Later, tools produce structured results that we can check field by field. An actual purchase needs a receipt from the supplier, not a different phrase in this checker.

## 5. Optional live request — 20 minutes maximum

The offline path works with no account. Use an endpoint and model you already have access to; this notebook makes no free-tier, price, model-availability or compatibility guarantee. A server on your laptop is not `localhost` in Colab.

To opt in, set `RUN_LIVE = True` and supply `CLASS_BASE_URL` and `CLASS_MODEL` in your notebook environment. If needed, store `CLASS_API_KEY` as a Colab secret or environment variable; do not paste a key in notebook source. The example expects a compatible `/chat/completions` response. Provider-specific settings may need adjustment. The main book documents its local Ollama path.

**PREDICT:** which observation would demonstrate a real model call? What would a timeout or a `finish_reason` of `length` establish? `max_tokens=20` may cause truncation, but it is not guaranteed to; record the actual envelope.

The socket timeout is not a total deadline. The response byte cap bounds the body we read. Endpoint failures are shown as failures before an explicitly labelled fixture fallback. Neither an available key nor Run all enables a live request.

In [ ]:
RUN_LIVE = False
BASE_URL = os.environ.get("CLASS_BASE_URL", "")
MODEL = os.environ.get("CLASS_MODEL", "")


def classroom_key():
    try:
        from google.colab import userdata

        return userdata.get("CLASS_API_KEY") or ""
    except Exception:
        return os.environ.get("CLASS_API_KEY", "")


class RefuseRedirects(HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        raise RuntimeError("Redirect refused; use the configured endpoint directly")


def live_call(body, *, base, key="", timeout=30, max_bytes=65_536):
    parts = urlsplit(base)
    if (
        parts.scheme not in {"https", "http"}
        or not parts.hostname
        or parts.query
        or parts.fragment
        or parts.username is not None
        or parts.password is not None
    ):
        raise ValueError("Use a plain endpoint base without credentials, query or fragment")
    if parts.scheme == "http" and parts.hostname not in {"localhost", "127.0.0.1", "::1"}:
        raise ValueError("Remote endpoints require HTTPS")
    headers = {"Content-Type": "application/json"}
    if key:
        headers["Authorization"] = "Bearer " + key
    request = Request(
        base.rstrip("/") + "/chat/completions", data=json.dumps(body).encode(), headers=headers
    )
    try:
        with build_opener(RefuseRedirects()).open(request, timeout=timeout) as response:
            raw = response.read(max_bytes + 1)
    except HTTPError as error:
        raise RuntimeError("HTTP failure " + str(error.code) + "; details withheld") from None
    except (URLError, OSError):
        raise RuntimeError("Connection failed; endpoint details withheld") from None
    if len(raw) > max_bytes:
        raise ValueError("response exceeded the byte ceiling")
    return json.loads(raw)


def run_trial(*, enabled=False, base="", model="", key="", call=live_call, max_tokens=256):
    if enabled:
        try:
            if not base or not model:
                raise ValueError("Select a base URL and model first")
            document = call(payload(SHOP, model, max_tokens), base=base, key=key)
            text = read_brief(document)
        except (OSError, RuntimeError, ValueError):
            print("LIVE ATTEMPT FAILED — no live result accepted; details withheld")
        else:
            print("LIVE MODEL RESPONSE — factual review still required")
            return {"mode": "live", "text": text, "flags": check_brief(text, SHOP)}
    print("OFFLINE RESPONSE FIXTURE — not evidence of a live connection")
    text = read_brief(OFFLINE_RESPONSE)
    return {"mode": "fixture", "text": text, "flags": check_brief(text, SHOP)}


trial = run_trial(
    enabled=RUN_LIVE, base=BASE_URL, model=MODEL, key=classroom_key() if RUN_LIVE else ""
)
print(trial)

If opted in, repeat three times and record `mode`, text, flags and manual factual review separately. Then change the output limit to 20 in a call to `run_trial`. Compare what actually happened, not a promised provider outcome. Stop configuring accounts after twenty minutes and continue with the fixture.

**Offline failure drill:** predict the mode returned when a transport fails. The fake transport below proves fallback behavior only.

In [ ]:
def unavailable(*args, **kwargs):
    raise RuntimeError("synthetic failure with private endpoint detail")


failed_trial = run_trial(
    enabled=True, base="https://example.invalid/v1", model="authored", call=unavailable
)
assert failed_trial["mode"] == "fixture"
print("Failure remained distinct from the fallback.")

## 6. Bind the response review to a snapshot — 10 minutes

**PREDICT:** if a delivery changes vanilla from 2 to 9 after serialization, which bytes change in the already-built request? Would our content heuristic notice? What should comparing snapshot stamps detect?

We stamp a copied snapshot and build its payload from that same copy. This local hash detects different content at review time; it does not prove freshness forever, authenticate a provider response, detect an intervening change followed by a return to the original content, or stop a change immediately after review. Later action boundaries need fresh authoritative records and versions.

In [ ]:
def snapshot_id(shop):
    return hashlib.sha256(json.dumps(shop, sort_keys=True).encode()).hexdigest()


def build(shop):
    snapshot = copy.deepcopy(shop)
    return {"snapshot": snapshot_id(snapshot), "body": payload(snapshot)}


def review_brief(built, document, current_shop):
    if built["snapshot"] != snapshot_id(current_shop):
        raise ValueError("shop changed since the request was built; request a fresh brief")
    text = read_brief(document)
    return {
        "status": "NEEDS_FACTUAL_REVIEW",
        "draft": text,
        "flags": check_brief(text, current_shop),
        "stock_facts": stock_facts(current_shop),
    }


changing_shop = copy.deepcopy(SHOP)
built = build(changing_shop)
print(review_brief(built, OFFLINE_RESPONSE, changing_shop)["status"])
changing_shop["products"][0]["on_hand"] = 9
assert json.loads(built["body"]["messages"][1]["content"])["products"][0]["on_hand"] == 2
try:
    review_brief(built, OFFLINE_RESPONSE, changing_shop)
except ValueError as error:
    print("STALE:", error)
else:
    raise AssertionError("Changed snapshot was not detected")
assert SHOP["products"][0]["on_hand"] == 2

## 7. Transfer task — 5 minutes, then homework

Lucy adds lime sorbet: SKU-LIME, zero tubs, reorder point four. **PREDICT:** what should an independent stock calculation report? What does the unchanged response fixture omit?

Work on a copy so rerunning the cell cannot append another product to the shared fixture. Add an assertion for Lime's needed quantity, then alter the fixture response and explain why that is an authored expected response, not model evidence.

In [ ]:
expanded_shop = copy.deepcopy(SHOP)
expanded_shop["products"].append(
    {"sku": "SKU-LIME", "name": "Lime", "on_hand": 0, "reorder_point": 4}
)
print(stock_facts(expanded_shop))
print(check_brief(read_brief(OFFLINE_RESPONSE), expanded_shop))
assert len(expanded_shop["products"]) == 4 and len(SHOP["products"]) == 3
# Your assertion: identify SKU-LIME and check its needed quantity against your prediction.

## Exit ticket — respond without running more code

1. A well-formed response says an order was placed. What evidence is missing?
2. Our warning check returns an empty list for `MISSED_LIE`. Why is it still false?
3. The live attempt failed but the notebook printed a fixture. Which mode belongs in your evidence log?
4. A snapshot stamp matches. Name two things that this does not prove.
5. What must a Chapter 2 stock tool read that a frozen prompt cannot discover?

Submit your written predictions, one intentionally malformed envelope and its rejection, one missed false claim, the stale-snapshot observation, and the Lime assertion. No paid endpoint is required. Next: build tools that read current records and calculate quantities in Python.